In [1]:
import os
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error




In [2]:
def seed_all(seed: int = 20):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)


seed_all(20)



In [3]:
DATA_DIR = "/kaggle/input/osic-pulmonary-fibrosis-progression"
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TEST_CSV = os.path.join(DATA_DIR, "test.csv")
SAMPLE_SUB_CSV = os.path.join(DATA_DIR, "sample_submission.csv")

train = pd.read_csv(TRAIN_CSV)
raw_test = pd.read_csv(TEST_CSV)
sample_sub = pd.read_csv(SAMPLE_SUB_CSV)

assert set(["Patient_Week", "FVC", "Confidence"]).issubset(sample_sub.columns)



In [4]:
X_prediction = sample_sub[["Patient_Week"]].copy()
X_prediction["Patient"] = X_prediction["Patient_Week"].str.extract(r"(.*)_.*")[0]
X_prediction["Weeks"] = (
    X_prediction["Patient_Week"].str.extract(r".*_(.*)")[0].astype(int)
)

X_prediction = X_prediction.merge(
    raw_test, on="Patient", how="left", suffixes=("", "_base")
)

X_prediction.rename(
    columns={"Weeks_base": "Base_week", "FVC": "Base_FVC"}, inplace=True
)

needed = [
    "Patient",
    "Patient_Week",
    "Weeks",
    "Base_week",
    "Base_FVC",
    "Percent",
    "Age",
    "Sex",
    "SmokingStatus",
]
missing = [c for c in needed if c not in X_prediction.columns]
if missing:
    raise RuntimeError(f"Missing required columns after merge: {missing}")



In [5]:
tr = train.copy()
tr = tr.sort_values(["Patient", "Weeks"]).reset_index(drop=True)

base = tr.groupby("Patient").first().reset_index()
base = base[["Patient", "Weeks", "FVC"]].rename(
    columns={"Weeks": "Base_week", "FVC": "Base_FVC"}
)

tr = tr.merge(base, on="Patient", how="left")
tr["delta_week"] = tr["Weeks"] - tr["Base_week"]


def prep_cats(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["Sex"] = df["Sex"].fillna("Unknown")
    df["SmokingStatus"] = df["SmokingStatus"].fillna("Unknown")
    return df


tr = prep_cats(tr)
X_prediction = prep_cats(X_prediction)




In [6]:
def fit_group_slopes(df: pd.DataFrame) -> pd.DataFrame:
    """
    Returns per-patient slope and intercept fitted on (delta_week, FVC),
    plus per-patient residual MAE as an uncertainty proxy.
    """
    rows = []
    for pid, g in df.groupby("Patient"):
        x = g["delta_week"].values.astype(float)
        y = g["FVC"].values.astype(float)
        if len(g) >= 2 and np.std(x) > 0:
            b, a = np.polyfit(x, y, 1)
            yhat = a + b * x
            mae = np.mean(np.abs(y - yhat))
        else:
            a = float(g["Base_FVC"].iloc[0])
            b = 0.0
            mae = np.mean(np.abs(y - a)) if len(g) else 200.0
        rows.append((pid, a, b, mae))
    return pd.DataFrame(rows, columns=["Patient", "a", "b", "mae"])


patient_fit = fit_group_slopes(tr)

tmp = tr.merge(patient_fit[["Patient", "b", "mae"]], on="Patient", how="left")
fallback = (
    tmp.groupby(["Sex", "SmokingStatus"], dropna=False)[["b", "mae"]]
    .median()
    .reset_index()
    .rename(columns={"b": "b_fb", "mae": "mae_fb"})
)




In [7]:
# Change (score-related): calibrate a single global confidence scale using OOF residuals.
# This aligns Confidence with the competition's Laplace log-likelihood, avoiding overconfident sigma.
def laplace_metric(y_true, y_pred, sigma):
    sigma_clip = np.maximum(sigma, 70.0)
    delta = np.minimum(np.abs(y_true - y_pred), 1000.0)
    return -np.sqrt(2.0) * delta / sigma_clip - np.log(np.sqrt(2.0) * sigma_clip)


gkf = GroupKFold(n_splits=5)
oof_pred = np.zeros(len(tr), dtype=float)
oof_mae_proxy = np.zeros(len(tr), dtype=float)

for fold, (trn_idx, val_idx) in enumerate(gkf.split(tr, groups=tr["Patient"]), 1):
    dtrn = tr.iloc[trn_idx].copy()
    dval = tr.iloc[val_idx].copy()

    pf = fit_group_slopes(dtrn)
    tmp2 = dtrn.merge(pf[["Patient", "b", "mae"]], on="Patient", how="left")
    fb = (
        tmp2.groupby(["Sex", "SmokingStatus"], dropna=False)[["b", "mae"]]
        .median()
        .reset_index()
        .rename(columns={"b": "b_fb", "mae": "mae_fb"})
    )

    dval = dval.merge(pf, on="Patient", how="left")
    dval = dval.merge(fb, on=["Sex", "SmokingStatus"], how="left")

    b_u = dval["b"].where(dval["b"].notna(), dval["b_fb"].fillna(0.0))
    a_u = dval["a"].where(dval["a"].notna(), dval["Base_FVC"].astype(float))
    fvc_hat = a_u + b_u * dval["delta_week"].astype(float)

    # same uncertainty proxy logic as in inference (per-patient mae, fallback to group median)
    mae_u = dval["mae"].where(dval["mae"].notna(), dval["mae_fb"].fillna(200.0))

    oof_pred[val_idx] = fvc_hat.values
    oof_mae_proxy[val_idx] = mae_u.values.astype(float)

# Choose a global multiplier k for sigma = max(70, k * mae_proxy) to maximize OOF metric.
# (Keeping this simple and deterministic; does not change the FVC model.)
y_true = tr["FVC"].values.astype(float)
resid = np.abs(y_true - oof_pred)

# Grid kept small for runtime; aims to improve from -8.06 toward ~-6.99 without over-optimizing.
k_grid = np.array([1.0, 1.25, 1.5, 1.75, 2.0, 2.25, 2.5, 3.0, 3.5, 4.0], dtype=float)
best_k = 1.5
best_score = -1e18
for k in k_grid:
    sigma = np.maximum(70.0, k * oof_mae_proxy)
    score = float(np.mean(laplace_metric(y_true, oof_pred, sigma)))
    if score > best_score:
        best_score = score
        best_k = float(k)

mae = mean_absolute_error(y_true, oof_pred)
print("OOF MAE (rough sanity check):", mae)
print(
    "OOF Laplace metric (approx, using sigma=k*mae_proxy):",
    best_score,
    "best_k:",
    best_k,
)



OOF MAE (rough sanity check): 143.95438900364462
OOF Laplace metric (approx, using sigma=k*mae_proxy): -6.657862617667346 best_k: 3.0


In [8]:
X_prediction = X_prediction.merge(patient_fit, on="Patient", how="left")
X_prediction = X_prediction.merge(fallback, on=["Sex", "SmokingStatus"], how="left")

b_used = X_prediction["b"].where(
    X_prediction["b"].notna(), X_prediction["b_fb"].fillna(0.0)
)
a_used = X_prediction["a"].where(
    X_prediction["a"].notna(), X_prediction["Base_FVC"].astype(float)
)

delta_week_pred = (X_prediction["Weeks"] - X_prediction["Base_week"]).astype(float)
fvc_pred = a_used + b_used * delta_week_pred

mae_used = X_prediction["mae"].where(
    X_prediction["mae"].notna(), X_prediction["mae_fb"].fillna(200.0)
)

# Change (score-related): use calibrated multiplier best_k for Confidence to be metric-friendly.
conf_pred = np.maximum(70.0, best_k * mae_used.values.astype(float))

X_prediction["FVC_pred"] = fvc_pred.values
X_prediction["Conf_pred"] = conf_pred



In [9]:
# Change (score-related): do NOT set an artificially tiny confidence for baseline points.
# The metric still clips sigma at 70, and tiny confidence does not help but can hurt calibration/robustness.
base_map = raw_test.set_index(["Patient", "Weeks"])["FVC"].to_dict()


def override_baseline(row):
    key = (row["Patient"], int(row["Weeks"]))
    if key in base_map:
        return float(base_map[key]), float(row["Conf_pred"])
    return float(row["FVC_pred"]), float(row["Conf_pred"])


overridden = X_prediction.apply(override_baseline, axis=1, result_type="expand")
X_prediction["FVC_final"] = overridden[0]
X_prediction["Confidence_final"] = overridden[1]

subm = sample_sub[["Patient_Week"]].merge(
    X_prediction[["Patient_Week", "FVC_final", "Confidence_final"]],
    on="Patient_Week",
    how="left",
)

subm["FVC"] = subm["FVC_final"].fillna(3020.0).astype(float)
subm["Confidence"] = subm["Confidence_final"].fillna(100.0).astype(float)

subm = subm[["Patient_Week", "FVC", "Confidence"]]
subm.to_csv("submission.csv", index=False)

print("Wrote submission.csv with shape:", subm.shape)
print(subm.head())
print("Confidence summary:", subm["Confidence"].describe())

Wrote submission.csv with shape: (1908, 3)
                   Patient_Week          FVC  Confidence
0  ID00126637202218610655908_-3  2460.478586  220.704651
1  ID00126637202218610655908_-2  2456.408178  220.704651
2  ID00126637202218610655908_-1  2452.337769  220.704651
3   ID00126637202218610655908_0  2448.267360  220.704651
4   ID00126637202218610655908_1  2444.196951  220.704651
Confidence summary: count    1908.000000
mean      203.224721
std        39.787659
min       107.750693
25%       158.185995
50%       220.704651
75%       220.704651
max       287.185639
Name: Confidence, dtype: float64
